In [0]:
WITH Count_vehicles AS (
  SELECT
    YEAR(road_entry_dt) AS road_entry_dt,
    COUNT(car_num) AS total_new_cars
  FROM
    transport_lakehouse.gold.fact_motorcycles
  GROUP BY
    YEAR(road_entry_dt)
  UNION ALL
  SELECT
    YEAR(road_entry_dt) AS road_entry_dt,
    COUNT(car_num) AS total_new_cars
  FROM
    transport_lakehouse.gold.fact_private_vehicles
  GROUP BY
    YEAR(road_entry_dt)
),
Final_Counts AS (
  SELECT
    road_entry_dt,
    SUM(total_new_cars) AS total_new_cars
  FROM
    Count_vehicles
  GROUP BY
    road_entry_dt
)
SELECT
  road_entry_dt,
  COALESCE(
    ROUND(
      100.0
        * (total_new_cars - LAG(total_new_cars) OVER (ORDER BY road_entry_dt))
        / LAG(total_new_cars) OVER (ORDER BY road_entry_dt),
      2
    ),
    0
  ) AS growth_pct
FROM
  Final_Counts
ORDER BY
  road_entry_dt DESC